In [1]:
import pyrealsense2 as rs
import numpy as np
import cv2
from RealSenseTools import list_realsense_devices

In [2]:
## Parameters of the videos
Width = 640
Height = 360
FPS = 60
VideoL = 60 #s 

In [3]:
SN = list_realsense_devices()

Found 2 connected RealSense devices:

--- Device 1 ---
  Name: Intel RealSense D555
  Serial Number: 353522301901
  Firmware Version: 7.56.19918.835
  Product ID: 0x0B56

--- Device 2 ---
  Name: Intel RealSense D555
  Serial Number: 353522300631
  Firmware Version: 7.56.19918.835
  Product ID: 0x0B56


In [4]:
# Serial numbers for the two cameras
serial1 = SN[0] # Replace with actual serial
serial2 = SN[1] # Replace with actual serial

# Initialize pipelines
pipe1 = rs.pipeline()
pipe2 = rs.pipeline()
config1 = rs.config()
config2 = rs.config()

# Enable streams
config1.enable_device(serial1)
config1.enable_stream(rs.stream.depth, Width, Height, rs.format.z16, FPS)
config1.enable_stream(rs.stream.color, Width, Height, rs.format.bgr8, FPS)

config2.enable_device(serial2)
config2.enable_stream(rs.stream.depth, Width, Height, rs.format.z16, FPS)
config2.enable_stream(rs.stream.color, Width, Height, rs.format.bgr8, FPS)

# Start pipelines
pipe1.start(config1)
pipe2.start(config2)

try:
    while True:
        # Get frames from both cameras
        frames1 = pipe1.wait_for_frames()
        frames2 = pipe2.wait_for_frames()
        color_frame1 = frames1.get_color_frame()
        color_frame2 = frames2.get_color_frame()
        if not color_frame1:
            continue

        # Convert images to numpy arrays
        color_image1 = np.asanyarray(color_frame1.get_data())
        color_image2 = np.asanyarray(color_frame2.get_data())
        
        # contatenate for displaying purposes
        frame_cat = cv2.hconcat([color_image1,color_image2])
        half = cv2.resize(frame_cat, (0, 0), fx = 0.5, fy = 0.5)
        cv2.imshow('frame', half)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    pipe1.stop()
    pipe2.stop()
    cv2.destroyAllWindows()